In [1]:
import pandas as pd
import sqlite3

In [2]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("database_cervezas.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [4]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

empleados
reparto
cervezas
bares


In [5]:
# crea las tablas

In [5]:
import sqlite3
connection = sqlite3.connect("database_cervezas.db")
crsr = connection.cursor()

In [6]:
query = '''
drop table if exists cervezas
'''

crsr.execute(query)

In [7]:
query = '''
drop table if exists bares
'''

crsr.execute(query)

In [8]:
crsr.execute('''
CREATE TABLE IF NOT EXISTS cervezas
("CodC",
"Envase",
"Capacidad",
"Stock")
''')
datos = [
  ('01', 'Botella', 0.2, 3600),
  ('02', 'Botella', 0.33, 1200),
  ('03', 'Lata', 0.33, 2400),
  ('04', 'Botella', 1, 288),
  ('05', 'Barril', 60, 30)
]
crsr.executemany("INSERT INTO cervezas VALUES (?, ?, ?, ?)",datos)
connection.commit()

In [9]:
cervezas = pd.read_sql_query("SELECT * FROM cervezas", connection)
display(cervezas)


,CodC,Envase,Capacidad,Stock
0,01,Botella,0.20,3600
1,02,Botella,0.33,1200
2,03,Lata,0.33,2400
3,04,Botella,1.00,288
4,05,Barril,60.00,30


In [10]:
crsr.execute('''
CREATE TABLE IF NOT EXISTS bares
("CodB",
"Nombre",
"Cif",
"Localidad")
''')
datos1 = [
  ('001', 'Stop', '11111111X', 'Villa Botijo'),
  ('002', 'Las Vegas', '22222222Y', 'Villa Botijo'),
  ('003', 'Club Social', '', 'Las Ranas'),
  ('004', 'Otra Ronda', '33333333Z', 'La Esponja'),
  ]

In [11]:
crsr.executemany("INSERT INTO bares VALUES (?, ?, ?, ?)", datos1)
connection.commit()

In [12]:
bares = pd.read_sql_query("SELECT * FROM bares", connection)
display(bares)

,CodB,Nombre,Cif,Localidad
0,001,Stop,11111111X,Villa Botijo
1,002,Las Vegas,22222222Y,Villa Botijo
2,003,Club Social,,Las Ranas
3,004,Otra Ronda,33333333Z,La Esponja


In [13]:
crsr.execute('''
CREATE TABLE IF NOT EXISTS empleados
("CodE" VARCHAR,
"Nombre" VARCHAR,
"Sueldo" INT)
''')

In [14]:

query = '''
INSERT OR REPLACE INTO empleados 
VALUES
("1", "Carlos Lopez", 120000),
("2", "Rosa Perez", 110000),
("3", "Luisa Garcia", 100000)
'''

crsr.execute(query)


In [15]:
empleados = pd.read_sql_query("SELECT * FROM empleados", connection)
display(empleados)

,CodE,Nombre,Sueldo
0,1,Carlos Lopez,120000
1,2,Rosa Perez,110000
2,3,Luisa Garcia,100000


In [16]:
query = '''
DROP TABLE IF EXISTS reparto
'''

crsr.execute(query)

In [17]:
crsr.execute('''
CREATE TABLE IF NOT EXISTS reparto
("CodE" VARCHAR,
"CodB" VARCHAR,
"CodC" VARCHAR,
"Fecha" DATE,
"Cantidad" INT)
''')

In [18]:
query = '''
INSERT OR REPLACE INTO reparto 
VALUES
("1", "001", "01", "10-21-05", 240),
("1", "001", "02", "10-21-05", 48),
("1", "002", "03", "10-22-05", 60),
("1", "004", "05", "10-22-05", 4),
("2", "002", "03", "10-22-05", 48),
("2", "002", "05", "10-23-05", 2),
("2", "004", "01", "10-23-05", 480),
("2", "004", "02", "10-24-05", 72),
("3", "003", "03", "10-24-05", 48),
("3", "003", "04", "10-25-05", 20)
'''

crsr.execute(query)

In [19]:
reparto = pd.read_sql_query("SELECT * FROM reparto", connection)
display(reparto)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,10-21-05,240
1,1,001,02,10-21-05,48
2,1,002,03,10-22-05,60
3,1,004,05,10-22-05,4
4,2,002,03,10-22-05,48
5,2,002,05,10-23-05,2
6,2,004,01,10-23-05,480
7,2,004,02,10-24-05,72
8,3,003,03,10-24-05,48
9,3,003,04,10-25-05,20


In [ ]:
# 1 Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana 
# del 17 al 23 de octubre de 2005.

query = '''

'''

sql_query(query)

In [20]:
query = '''
SELECT DISTINCT e.Nombre
FROM empleados e
INNER JOIN reparto r ON e.CodE = r.CodE
INNER JOIN bares b ON r.CodB = b.CodB
WHERE b.Nombre = "Stop"
AND r.Fecha BETWEEN "10-17-05" and "10-23-05"
'''

sql_query(query)

,Nombre
0,Carlos Lopez


In [ ]:
# 2 Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y
# capacidad inferior a 1 litro, ordenados por localidad

query = '''

'''

sql_query(query)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [21]:
query = '''
SELECT DISTINCT
       b.CIF,
       b.Nombre
FROM bares b
INNER JOIN reparto r ON b.CodB = r.CodB
INNER JOIN cervezas c ON r.CodC = c.CodC
WHERE c.Envase = 'Botella'
  AND c.Capacidad < 1
ORDER BY b.Localidad;
'''
sql_query(query)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [ ]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad)

query = '''

'''

sql_query(query)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


In [33]:
query = '''
SELECT
    b.Nombre,
    c.Envase,
    c.Capacidad,
    r.Fecha,
    r.Cantidad
FROM reparto r
INNER JOIN bares b ON r.CodB = b.CodB
INNER JOIN cervezas c ON r.CodC = c.CodC
ORDER BY b.Nombre, r.Fecha;
'''
sql_query(query)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Club Social,Lata,0.33,10-24-05,48
1,Club Social,Botella,1.00,10-25-05,20
2,Las Vegas,Lata,0.33,10-22-05,60
3,Las Vegas,Lata,0.33,10-22-05,48
4,Las Vegas,Barril,60.00,10-23-05,2
5,Otra Ronda,Barril,60.00,10-22-05,4
6,Otra Ronda,Botella,0.20,10-23-05,480
7,Otra Ronda,Botella,0.33,10-24-05,72
8,Stop,Botella,0.20,10-21-05,240
9,Stop,Botella,0.33,10-21-05,48


In [ ]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 o
# 0.33

query = '''

'''
sql_query(query)

,Nombre,Envase,Capacidad
0,Stop,Botella,0.20
1,Stop,Botella,0.33
2,Otra Ronda,Botella,0.20
3,Otra Ronda,Botella,0.33


In [ ]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con
# envase botella.

query = '''

'''
sql_query(query)

,Nombre bar,Nombre empleado,Envase
0,Stop,Carlos Lopez,Botella


In [ ]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa
# Botijo.

query = '''

'''
sql_query(query)

,CodE,Nombre,Viajes
0,1,Carlos Lopez,1
1,2,Rosa Perez,2
2,3,Luisa Garcia,2


In [ ]:
query = '''

'''
sql_query(query)

,Nombre,Localidad,litros
0,Otra Ronda,La Esponja,359.76


In [ ]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''

'''
sql_query(query)

,Nombre,Localidad,litros
0,Otra Ronda,La Esponja,359.76


In [ ]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y
# capacidad menor que 1 litro.

query = '''

'''
sql_query(query)

,Nombre
0,Stop
1,Otra Ronda


In [ ]:
#9. Subir un 5% el sueldo del empleado que más días haya trabajado.

query = '''

'''
sql_query(query)

,Dias_trabajados,CodE,Nombre
0,3,2,Rosa Perez
1,2,3,Luisa Garcia
2,2,1,Carlos Lopez


In [ ]:
query = '''

'''
sql_query(query)

,t.Sueldo * 1.05
0,115500.0


In [ ]:
query = '''
   
'''
sql_query(query)

,CodE
0,2


In [ ]:
query = '''

'''
crsr.execute(query)

In [ ]:
query = '''

'''
sql_query(query)

,CodE,Nombre,Sueldo
0,2,Rosa Perez,115500
